# Рубежный контроль №1 — вариант 11

Задание: [TMO_RK_1](https://github.com/ugapanyuk/courses_current/wiki/TMO_RK_1).

- Вариант 11 → задача №2, датасет №3.
- Доп. требование (РТ5-61Б): `jointplot`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
%matplotlib inline

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## Данные и искусственные пропуски

Если в датасете нет нужных признаков/пропусков — их можно создать искусственно (примечание в методичке).

In [ ]:
n = 600
cities = np.array(["Moscow", "SPb", "Kazan", "Novosibirsk", "Other"], dtype=object)
genders = np.array(["female", "male"], dtype=object)

df = pd.DataFrame({
    "gender": rng.choice(genders, size=n, p=[0.52, 0.48]),
    "city": rng.choice(cities, size=n, p=[0.35, 0.2, 0.15, 0.1, 0.2]),
    "age": np.clip(rng.normal(36, 11, size=n), 18, 75),
    "income": np.clip(rng.lognormal(10.5, 0.35, size=n), 20_000, 800_000),
})

miss_city = rng.choice(df.index, size=int(n * 0.10), replace=False)
miss_age = rng.choice(df.index.difference(miss_city), size=int(n * 0.12), replace=False)
df.loc[miss_city, "city"] = np.nan
df.loc[miss_age, "age"] = np.nan

display(df.head())
display(df.isna().sum())

## Jointplot (РТ5-61Б)

Построим `jointplot` для пары числовых признаков `age` и `income` (строки с NaN удаляем только для графика).

In [ ]:
tmp = df[["age", "income"]].dropna()
g = sns.jointplot(data=tmp, x="age", y="income", kind="scatter", height=6, alpha=0.6)
g.fig.suptitle("Jointplot: age vs income", y=1.02)
plt.show()

## Обработка пропусков (задача №2)

- `city` (категориальный): заполняем **модой**.
- `age` (числовой): заполняем **медианой**.

In [ ]:
df_clean = df.copy()
city_mode = df_clean["city"].mode(dropna=True)[0]
age_median = float(df_clean["age"].median(skipna=True))

df_clean["city"] = df_clean["city"].fillna(city_mode)
df_clean["age"] = df_clean["age"].fillna(age_median)

print("city mode:", city_mode)
print("age median:", round(age_median, 3))
display(df_clean.isna().sum())

## Ответы

1. **Способы обработки пропусков:**
   - для категориального признака `city` — заполнение **модой**;
   - для количественного признака `age` — заполнение **медианой**.

2. **Какие признаки использовать для моделей и почему:**
   - `age`, `income` — числовые, напрямую подходят большинству моделей;
   - `gender`, `city` — категориальные, требуют кодирования (например, one-hot) и добавляют информацию о сегментах.
   
Удаление строк с пропусками снижает объём данных, поэтому здесь выбрана импутация простыми и интерпретируемыми стратегиями.